# Command Classifier Evaluation

This notebook evaluates the trained Simple Neural Network command classifier.

**Metrics computed:**
- Accuracy, Precision, Recall, F1 (per-class and macro)
- Confusion matrix visualization
- Inference latency benchmarks (CPU)


In [ ]:
import sys
sys.path.append('..')

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import torch
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from src.dataset import DataProcessor
from src.inference import load_classifier

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Imports loaded successfully!")


## 1. Load Data and Model


In [ ]:
# Load the classifier
print("Loading classifier...")
classifier = load_classifier()

# Load full dataset for evaluation
print("\nLoading dataset...")
processor = DataProcessor()
train_dataset, val_dataset = processor.prepare_data(test_size=0.2, random_state=42)

class_names = processor.get_class_names()
print(f"\nClasses: {class_names}")


## 2. Evaluate on Validation Set


In [ ]:
# Get predictions on validation set
val_texts = val_dataset.texts
val_labels = val_dataset.labels.numpy()

print(f"Evaluating on {len(val_texts)} validation samples...")

predictions = []
confidences = []

for text in val_texts:
    result = classifier.classify(text)
    predictions.append(processor.intent_to_idx[result['intent']])
    confidences.append(result['confidence'])

predictions = np.array(predictions)
confidences = np.array(confidences)

print("Evaluation complete!")


In [ ]:
# Calculate metrics
accuracy = accuracy_score(val_labels, predictions)
precision, recall, f1, support = precision_recall_fscore_support(
    val_labels, predictions, average=None
)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    val_labels, predictions, average='macro'
)

print("=" * 60)
print("OVERALL METRICS")
print("=" * 60)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {macro_precision:.4f} (macro)")
print(f"Recall:    {macro_recall:.4f} (macro)")
print(f"F1 Score:  {macro_f1:.4f} (macro)")
print(f"\nMean Confidence: {np.mean(confidences):.4f}")


In [ ]:
# Per-class metrics
print("\n" + "=" * 60)
print("PER-CLASS METRICS")
print("=" * 60)
print(classification_report(val_labels, predictions, target_names=class_names))


## 3. Confusion Matrix


In [ ]:
# Compute confusion matrix
cm = confusion_matrix(val_labels, predictions)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix - Command Classification')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../checkpoints/confusion_matrix.png', dpi=150)
plt.show()

print("Confusion matrix saved to checkpoints/confusion_matrix.png")


## 4. Latency Benchmarks


In [ ]:
# Test commands for benchmarking
test_commands = [
    "Take me to the oxygen tank",
    "Show diagnostic info for this panel",
    "Increase temperature by 5 degrees",
    "What is the current pressure reading?",
    "Mark this as critical",
    "Record maintenance completed",
    "How do I replace the filter?",
    "Stop current action"
]

# Run benchmark
print("Running latency benchmark...")
benchmark_results = classifier.benchmark(test_commands, num_runs=100)

print("\n" + "=" * 60)
print("LATENCY BENCHMARK RESULTS")
print("=" * 60)
print(f"Total runs: {benchmark_results['num_runs']}")
print(f"\nEmbedding Time:")
print(f"  Mean: {benchmark_results['embedding']['mean_ms']:.2f}ms")
print(f"  Std:  {benchmark_results['embedding']['std_ms']:.2f}ms")
print(f"  P95:  {benchmark_results['embedding']['p95_ms']:.2f}ms")
print(f"\nInference Time (NN only):")
print(f"  Mean: {benchmark_results['inference']['mean_ms']:.2f}ms")
print(f"  Std:  {benchmark_results['inference']['std_ms']:.2f}ms")
print(f"  P95:  {benchmark_results['inference']['p95_ms']:.2f}ms")
print(f"\nTotal Time (Embedding + Inference):")
print(f"  Mean: {benchmark_results['total']['mean_ms']:.2f}ms")
print(f"  Std:  {benchmark_results['total']['std_ms']:.2f}ms")
print(f"  P95:  {benchmark_results['total']['p95_ms']:.2f}ms")


## 5. Training History


In [ ]:
# Load training history
history_path = Path('../checkpoints/training_history.json')

if history_path.exists():
    with open(history_path, 'r') as f:
        history = json.load(f)
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Loss plot
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train')
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy plot
    axes[1].plot(epochs, history['train_acc'], 'b-', label='Train')
    axes[1].plot(epochs, history['val_acc'], 'r-', label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Training Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../checkpoints/training_curves.png', dpi=150)
    plt.show()
    
    print(f"Final Training Accuracy: {history['train_acc'][-1]:.4f}")
    print(f"Final Validation Accuracy: {history['val_acc'][-1]:.4f}")
else:
    print("No training history found. Run training first.")


## 6. Model Summary


In [ ]:
# Model architecture summary
print("=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
print(classifier.model.get_architecture_summary())

print("\n" + "=" * 60)
print("DETAILED LAYER INFO")
print("=" * 60)
for name, param in classifier.model.named_parameters():
    print(f"{name:30s} | Shape: {str(list(param.shape)):20s} | Params: {param.numel():,}")


## 7. Save Evaluation Results


In [ ]:
# Save comprehensive evaluation results
eval_results = {
    "overall_metrics": {
        "accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "mean_confidence": float(np.mean(confidences))
    },
    "per_class_metrics": {
        class_name: {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i])
        }
        for i, class_name in enumerate(class_names)
    },
    "latency_benchmarks": benchmark_results,
    "model_info": {
        "embedding_dim": classifier.model.embedding_dim,
        "hidden_dims": classifier.model.hidden_dims,
        "num_classes": classifier.model.num_classes,
        "total_parameters": classifier.model.count_parameters()
    }
}

with open('../checkpoints/evaluation_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)

print("Evaluation results saved to checkpoints/evaluation_results.json")


## 8. Interactive Testing


In [ ]:
# Test individual commands
test_examples = [
    "Navigate to the medical bay",
    "What's the oxygen level?",
    "Set fan speed to maximum",
    "Help me troubleshoot this error",
    "Stop everything"
]

print("Sample Predictions:")
print("=" * 60)

for cmd in test_examples:
    result = classifier.classify(cmd, return_all_probs=True)
    print(f"\nCommand: \"{cmd}\"")
    print(f"  Predicted: {result['intent']}")
    print(f"  Confidence: {result['confidence']:.2%}")
